In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['text.usetex'] = True
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}"

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx
import diffrax

---

In [ ]:
from exciting_environments import EnvironmentRegistry
from exciting_environments.utils import MinMaxNormalization

In [ ]:
from dmpe.utils.density_estimation import build_grid
from dmpe.utils.env_utils.fluid_tank_utils import setup_env as setup_fluid_tank_env
from dmpe.utils.env_utils.pendulum_utils import setup_env as setup_pendulum_env
from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

## Inspect vectorfield:

- show the dynamics vectorfield (=sensitivity?)
- show the curvature
- build a function: `vector_field = func(env)` where the output is dependent on both the states and the inputs
    - plotting is not so important, it is more about using the dynamics to produce a target distribution
    - rework `estimate_dynamics_grid`

#### 3D Pendulum:

In [ ]:
from admpe.vector_fields import VectorField, DynamicsVectorField, CurvatureVectorField

In [ ]:
env, penalty_function, featurize, env_params = setup_pendulum_env()
dvf = DynamicsVectorField.from_env(env, 15)
dvf

In [ ]:
fig, ax = dvf.visualize(dims=(0, 1), aggregate="slice", fixed_value=0.5)

#### 2D fluid_tank (1D state - 1D input):

In [ ]:
env, penalty_function, featurize, env_params = setup_fluid_tank_env()
dvf = DynamicsVectorField.from_env(env, 15)
dvf

In [ ]:
fig, ax = dvf.visualize(dims=(0, 1))

#### Cart-pole (4D state - 1D input):

In [ ]:
env, penalty_function, featurize, env_params = setup_cart_pole_env()
dvf = DynamicsVectorField.from_env(env, 15)
dvf

In [ ]:
fig, ax = dvf.visualize(dims=(1, 4))

## Build target distribution based on dynamics vector field:

**You have to watch out that the orientation of dims fits to what you are optimizing for. (match dyn_vec_f with target and data distribution)**

In [ ]:
from dmpe.utils.density_estimation import DensityEstimate, get_uniform_target_distribution
from dmpe.excitation.excitation_utils import soft_penalty

In [ ]:
from admpe.adaptive_densities import TargetDensity

In [ ]:
env, penalty_function, featurize, env_params = setup_pendulum_env()
dvf = DynamicsVectorField.from_env(env, 40)
dvf

target_density = TargetDensity.from_vector_field(
    vf=dvf,
    bandwidth=0.1,
    penalty_function=penalty_function,
    consider_action_distribution=True,
    obs_dim=env.obs_dim,
    act_dim=env.action_dim,
)

In [ ]:
target_density.visualize()

- I think the density going so low in parts cannot be correct. There must be a less invasive mapping.
- Maybe it makes sense to just run the algorithm and see what happens?

## Build curvature vectorfield

Away from curvature, towards

$$ \large \frac{\mathrm{d}^2}{\mathrm{d}t^2} \boldsymbol{x} (t) = \frac{\mathrm{d}}{\mathrm{d}t} \boldsymbol{f}$$

But that is not really what I am looking for? Or is it...

- local Lyapunov exponents